# 第11课：七种策略最终比较与Robustness Check

本课完成Python模型的策略分析部分：

1. 补上`Updated 50%`策略；
2. 在同一批10,000个情景中比较全部七种策略；
3. 应用提前声明的选择规则；
4. 把July price beta设为0重新运行，检查推荐是否改变。

完成本课后，我们才可以给出模型范围内的最终策略推荐。

## 0. 七种策略

| 类别 | 策略 |
|---|---|
| Unhedged | Fixed 0% |
| Fixed | Fixed 25%、50%、75%、100% |
| Updated-production | Updated 50% |
| Weather adaptive | Weather-signal 25/50/75% |

所有策略面对相同的yield、futures、basis和cash-price情景。

## 1. Updated 50%是什么？

Updated 50%与Fixed 50%的区别不是目标比例，而是计算目标bushels时使用的信息不同：

- 3月：按照preseason expected production的50%建立初始头寸；
- 7月：仍然保持50%比例，但改用July updated production forecast重新计算合约数。

$$N_0=Round\left(\frac{0.50\times ExpectedYield\times Acres}{5{,}000}\right)$$

$$N_{July}=Round\left(\frac{0.50\times JulyYieldForecast\times Acres}{5{,}000}\right)$$

$$\Delta N=N_{July}-N_0$$

因此它会小幅调整合约数量，但不会把目标比例改成25%或75%。

## 2. 最终选择标准

提前锁定的规则保持不变：

1. **Primary objective**：在eligible strategies中最大化CVaR 5% profit per acre；
2. **Expected-profit constraint**：平均利润不得比unhedged mean低超过$5\%\times|UnhedgedMean|$；
3. **Over-hedging constraint**：over-hedging probability不得超过10%；
4. 如果CVaR 5%相同，用更高expected profit打破平局。

其他指标仍会完整报告，但不会临时改变决策规则。

## 3. 导入工具并锁定全部参数

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
FARM_ACRES = 1_000
CONTRACT_SIZE_BUSHELS = 5_000
STRATEGIES = [
    'Fixed 0%',
    'Fixed 25%',
    'Fixed 50%',
    'Fixed 75%',
    'Fixed 100%',
    'Updated 50%',
    'Weather-signal 25/50/75%',
]

PDSI_LOW_THRESHOLD = -0.2166666666666667
PDSI_HIGH_THRESHOLD = 2.3099999999999996

TRANSACTION_COST_PER_CONTRACT_SIDE = 25.00
INITIAL_MARGIN_PER_CONTRACT = 2_500.00
MARGIN_FINANCING_RATE_ANNUAL = 0.06
DAYS_PRESEASON_TO_JULY = 136
DAYS_JULY_TO_HARVEST = 108
MARGIN_LIQUIDITY_RESERVE = 50_000.00

LOWER_TAIL_PROBABILITY = 0.05
MAX_MEAN_PROFIT_SHORTFALL_FRACTION = 0.05
MAX_OVERHEDGE_PROBABILITY = 0.10
Z_95 = 1.96

PRICE_FLOOR = 1.50
JULY_PRICE_INTERCEPT = -0.05375000000000204
MAIN_JULY_PRICE_BETA = -0.019741301483521958

print('Strategies:', len(STRATEGIES))
print('Scenarios per strategy:', N_SIMULATIONS)
print('Main July price beta:', MAIN_JULY_PRICE_BETA)

## 4. 读取第7课共同情景

本Notebook只需要一份基础共同情景，随后会在内部重新评价全部策略。

In [ ]:
candidate_paths = [
    Path.cwd() / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
    Path.cwd().parent / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
]

lesson7_path = next((p for p in candidate_paths if p.exists()), None)
if lesson7_path is None:
    raise FileNotFoundError('没有找到第7课共同情景CSV。请先运行Lesson_07。')

main_scenarios = pd.read_csv(lesson7_path)

required_columns = [
    'scenario_id',
    'july_pdsi',
    'preseason_expected_yield_bu_per_acre',
    'july_yield_forecast_bu_per_acre',
    'actual_production_bushels',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'july_price_residual_usd_per_bushel',
    'cash_revenue_usd',
    'production_cost_usd',
]
missing = [c for c in required_columns if c not in main_scenarios.columns]

assert not missing, f'缺少列: {missing}'
assert len(main_scenarios) == N_SIMULATIONS
assert main_scenarios['scenario_id'].is_unique
assert main_scenarios[required_columns].isna().sum().sum() == 0

print('共同情景检查通过:', lesson7_path)

## 5. 建立Contract Rounding函数

In [ ]:
def rounded_contracts(quantity_bushels, contract_size=CONTRACT_SIZE_BUSHELS):
    quantity_bushels = np.asarray(quantity_bushels, dtype=float)
    return np.floor(np.maximum(quantity_bushels, 0.0) / contract_size + 0.5).astype(int)

## 6. 建立Strategy Position函数

这个函数只决定：

- 3月initial contracts；
- 7月target ratio；
- July adjustment；
- Harvest前final contracts。

损益计算留给下一函数，避免把“决策规则”和“会计结果”混在一起。

In [ ]:
def strategy_positions(strategy, scenario_data):
    expected_yield = scenario_data['preseason_expected_yield_bu_per_acre'].to_numpy()
    updated_yield = scenario_data['july_yield_forecast_bu_per_acre'].to_numpy()

    if strategy.startswith('Fixed '):
        ratio = float(strategy.split()[1].rstrip('%')) / 100.0
        initial = rounded_contracts(ratio * expected_yield * FARM_ACRES)
        adjustment = np.zeros(len(scenario_data), dtype=int)
        july_ratio = np.full(len(scenario_data), ratio)
        return initial, adjustment, july_ratio

    # 两个动态策略都从3月50%开始
    initial = rounded_contracts(0.50 * expected_yield * FARM_ACRES)

    if strategy == 'Updated 50%':
        july_ratio = np.full(len(scenario_data), 0.50)
    elif strategy == 'Weather-signal 25/50/75%':
        pdsi = scenario_data['july_pdsi'].to_numpy()
        july_ratio = np.select(
            [
                pdsi < PDSI_LOW_THRESHOLD,
                pdsi > PDSI_HIGH_THRESHOLD,
            ],
            [0.25, 0.75],
            default=0.50,
        )
    else:
        raise ValueError(f'Unknown strategy: {strategy}')

    final_contracts = rounded_contracts(
        july_ratio * updated_yield * FARM_ACRES
    )
    adjustment = final_contracts - initial
    return initial, adjustment, july_ratio

## 7. 建立统一Strategy Evaluation函数

所有策略必须用同样的P&L、transaction-cost、margin-cost和profit公式。

In [ ]:
def evaluate_strategy(strategy, scenario_data):
    initial, adjustment, july_ratio = strategy_positions(strategy, scenario_data)
    final_contracts = initial + adjustment

    f0 = scenario_data['preseason_futures_usd_per_bushel'].to_numpy()
    f_july = scenario_data['july_futures_usd_per_bushel'].to_numpy()
    f_harvest = scenario_data['harvest_futures_usd_per_bushel'].to_numpy()

    initial_futures_pnl = (
        initial * CONTRACT_SIZE_BUSHELS * (f0 - f_harvest)
    )
    july_adjustment_pnl = (
        adjustment * CONTRACT_SIZE_BUSHELS * (f_july - f_harvest)
    )
    total_futures_pnl = initial_futures_pnl + july_adjustment_pnl

    transaction_sides = (
        np.abs(initial) + np.abs(adjustment) + np.abs(final_contracts)
    )
    transaction_cost = transaction_sides * TRANSACTION_COST_PER_CONTRACT_SIDE

    margin_financing_cost = (
        INITIAL_MARGIN_PER_CONTRACT
        * MARGIN_FINANCING_RATE_ANNUAL
        * (
            np.abs(initial) * DAYS_PRESEASON_TO_JULY / 365.0
            + np.abs(final_contracts) * DAYS_JULY_TO_HARVEST / 365.0
        )
    )

    cash_revenue = scenario_data['cash_revenue_usd'].to_numpy()
    production_cost = scenario_data['production_cost_usd'].to_numpy()
    gross_revenue_after_hedge = (
        cash_revenue + total_futures_pnl
        - transaction_cost - margin_financing_cost
    )
    profit = gross_revenue_after_hedge - production_cost

    actual_production = scenario_data['actual_production_bushels'].to_numpy()
    final_hedged_bushels = final_contracts * CONTRACT_SIZE_BUSHELS
    overhedged = final_hedged_bushels > actual_production

    july_mtm = initial * CONTRACT_SIZE_BUSHELS * (f0 - f_july)
    harvest_segment_mtm = (
        final_contracts * CONTRACT_SIZE_BUSHELS * (f_july - f_harvest)
    )
    margin_call_proxy = (
        ((-july_mtm) > MARGIN_LIQUIDITY_RESERVE)
        | ((-harvest_segment_mtm) > MARGIN_LIQUIDITY_RESERVE)
    )

    initial_ratio = (
        float(strategy.split()[1].rstrip('%')) / 100.0
        if strategy.startswith('Fixed ')
        else 0.50
    )

    return pd.DataFrame({
        'scenario_id': scenario_data['scenario_id'].to_numpy(),
        'strategy': strategy,
        'initial_hedge_ratio': initial_ratio,
        'july_target_hedge_ratio': july_ratio,
        'initial_contracts': initial,
        'july_adjustment_contracts': adjustment,
        'final_contracts': final_contracts,
        'actual_production_bushels': actual_production,
        'final_hedged_bushels': final_hedged_bushels,
        'cash_revenue_usd': cash_revenue,
        'initial_futures_pnl_usd': initial_futures_pnl,
        'july_adjustment_pnl_usd': july_adjustment_pnl,
        'total_futures_pnl_usd': total_futures_pnl,
        'transaction_cost_usd': transaction_cost,
        'margin_financing_cost_usd': margin_financing_cost,
        'gross_revenue_after_hedge_usd': gross_revenue_after_hedge,
        'profit_usd': profit,
        'profit_usd_per_acre': profit / FARM_ACRES,
        'overhedged': overhedged,
        'margin_call_proxy': margin_call_proxy,
    })

## 8. 建立Summary与Recommendation函数

In [ ]:
def summarize_all(results):
    rows = []
    for strategy, group in results.groupby('strategy', sort=False):
        profit = group['profit_usd_per_acre']
        p5 = profit.quantile(LOWER_TAIL_PROBABILITY)
        standard_error = profit.std(ddof=1) / np.sqrt(len(profit))
        rows.append({
            'strategy': strategy,
            'Expected Profit ($/acre)': profit.mean(),
            'Profit Std Dev ($/acre)': profit.std(ddof=1),
            'Probability Profit < 0': (profit < 0).mean(),
            'P5 Profit ($/acre)': p5,
            'CVaR 5% Profit ($/acre)': profit[profit <= p5].mean(),
            'Probability Overhedged': group['overhedged'].mean(),
            'Probability Margin Call Proxy': group['margin_call_proxy'].mean(),
            'Avg Transaction Cost ($/acre)': group['transaction_cost_usd'].mean() / FARM_ACRES,
            'Avg Margin Financing Cost ($/acre)': group['margin_financing_cost_usd'].mean() / FARM_ACRES,
            'Mean 95% CI Low ($/acre)': profit.mean() - Z_95 * standard_error,
            'Mean 95% CI High ($/acre)': profit.mean() + Z_95 * standard_error,
        })
    return pd.DataFrame(rows)


def choose_preferred(summary):
    scored = summary.copy()
    unhedged_mean = scored.loc[
        scored['strategy'] == 'Fixed 0%',
        'Expected Profit ($/acre)',
    ].iloc[0]
    minimum_mean = (
        unhedged_mean
        - MAX_MEAN_PROFIT_SHORTFALL_FRACTION * abs(unhedged_mean)
    )

    scored['Pass Expected-Profit Rule'] = (
        scored['Expected Profit ($/acre)'] >= minimum_mean
    )
    scored['Pass Overhedge Rule'] = (
        scored['Probability Overhedged'] <= MAX_OVERHEDGE_PROBABILITY
    )
    scored['Eligible'] = (
        scored['Pass Expected-Profit Rule']
        & scored['Pass Overhedge Rule']
    )

    eligible = scored[scored['Eligible']]
    preferred = (
        eligible.sort_values(
            ['CVaR 5% Profit ($/acre)', 'Expected Profit ($/acre)'],
            ascending=[False, False],
        )
        .iloc[0]['strategy']
    )
    scored['Preferred'] = scored['strategy'] == preferred
    return scored, preferred, minimum_mean

## 9. 运行主模型的全部七种策略

In [ ]:
main_results = pd.concat(
    [evaluate_strategy(strategy, main_scenarios) for strategy in STRATEGIES],
    ignore_index=True,
)

main_summary = summarize_all(main_results)
main_summary, main_preferred, main_minimum_mean = choose_preferred(main_summary)

print('Main-model result rows:', len(main_results))
print('Main-model preferred strategy:', main_preferred)
print(main_summary.round(4).to_string(index=False))

## 10. 查看Updated 50%的实际调整

它只会围绕21份初始合约做小幅调整。

In [ ]:
updated50 = main_results[main_results['strategy'] == 'Updated 50%']

print('Updated 50% July adjustments:')
print(updated50['july_adjustment_contracts'].value_counts().sort_index().to_string())
print()
print('Updated 50% final contracts:')
print(updated50['final_contracts'].value_counts().sort_index().to_string())

assert (updated50['initial_contracts'] == 21).all()
assert set(updated50['july_adjustment_contracts'].unique()) == {-1, 0, 1}

## 11. 主模型最终选择

Fixed 100%虽然expected profit最高，但over-hedging probability超过10%，不合格。其余策略中，Fixed 75%的CVaR 5%最高。

In [ ]:
print(main_summary[[
    'strategy',
    'Expected Profit ($/acre)',
    'Profit Std Dev ($/acre)',
    'P5 Profit ($/acre)',
    'CVaR 5% Profit ($/acre)',
    'Probability Overhedged',
    'Eligible',
    'Preferred',
]].round(4).to_string(index=False))

print()
print('Minimum eligible expected profit:', round(main_minimum_mean, 4))
print('Main-model recommendation:', main_preferred)

## 12. 为什么需要July-beta Robustness Check？

第4课的July价格模型为：

$$\Delta F_{July}=\alpha_J+\gamma_J(JulyYieldSignal)+e_J$$

其中$γ_J\approx-0.01974$，但模型$R^2\approx0.022$，解释力很弱。

Robustness test把$γ_J$设为0：

$$F_{July}^{robust}=\max(1.50,F_0+\alpha_J+e_J)$$

Harvest futures、final yield、basis和cash price保持不变。这样检验最终推荐是否依赖较弱的July signal coefficient。

## 13. 建立July beta = 0的情景

In [ ]:
robust_scenarios = main_scenarios.copy()

robust_july_futures = np.maximum(
    PRICE_FLOOR,
    robust_scenarios['preseason_futures_usd_per_bushel'].to_numpy()
    + JULY_PRICE_INTERCEPT
    + robust_scenarios['july_price_residual_usd_per_bushel'].to_numpy()
)

robust_scenarios['july_futures_usd_per_bushel'] = robust_july_futures

price_comparison = pd.DataFrame({
    'Main July Futures': main_scenarios['july_futures_usd_per_bushel'],
    'Robust July Futures': robust_july_futures,
})

print(price_comparison.describe(percentiles=[0.05, 0.50, 0.95]).round(4).to_string())
print('Robust price-floor count:', int((robust_july_futures <= PRICE_FLOOR + 1e-12).sum()))

## 14. 在Robustness情景下重新运行全部七种策略

In [ ]:
robust_results = pd.concat(
    [evaluate_strategy(strategy, robust_scenarios) for strategy in STRATEGIES],
    ignore_index=True,
)

robust_summary = summarize_all(robust_results)
robust_summary, robust_preferred, robust_minimum_mean = choose_preferred(robust_summary)

print('Robust-model result rows:', len(robust_results))
print('Robust-model preferred strategy:', robust_preferred)
print(robust_summary.round(4).to_string(index=False))

## 15. 主模型与Robustness结果并排比较

In [ ]:
comparison = main_summary[[
    'strategy',
    'Expected Profit ($/acre)',
    'CVaR 5% Profit ($/acre)',
    'Probability Margin Call Proxy',
    'Preferred',
]].merge(
    robust_summary[[
        'strategy',
        'Expected Profit ($/acre)',
        'CVaR 5% Profit ($/acre)',
        'Probability Margin Call Proxy',
        'Preferred',
    ]],
    on='strategy',
    suffixes=(' - Main', ' - Beta0'),
)

print(comparison.round(4).to_string(index=False))

## 16. Robustness结论

Fixed strategies的final profit不依赖July price，因为它们不在7月调整；但margin-call proxy可能变化。

Updated 50%和Weather-signal策略的July adjustment P&L会变化，因此其profit metrics也会变化。关键问题是首选策略是否改变。

In [ ]:
print('Main preferred strategy:', main_preferred)
print('Beta=0 preferred strategy:', robust_preferred)
print('Recommendation unchanged:', main_preferred == robust_preferred)

weather_main = main_summary[main_summary['strategy'] == 'Weather-signal 25/50/75%'].iloc[0]
weather_robust = robust_summary[robust_summary['strategy'] == 'Weather-signal 25/50/75%'].iloc[0]

print()
print('Weather-signal mean-profit change:', round(
    weather_robust['Expected Profit ($/acre)'] - weather_main['Expected Profit ($/acre)'], 4
))
print('Weather-signal CVaR change:', round(
    weather_robust['CVaR 5% Profit ($/acre)'] - weather_main['CVaR 5% Profit ($/acre)'], 4
))

## 17. 图1：主模型七策略Mean、P5与CVaR

In [ ]:
plt.figure(figsize=(12, 5))
x = main_summary['strategy']
plt.plot(x, main_summary['Expected Profit ($/acre)'], marker='o', label='Expected Profit')
plt.plot(x, main_summary['P5 Profit ($/acre)'], marker='o', label='P5')
plt.plot(x, main_summary['CVaR 5% Profit ($/acre)'], marker='o', label='CVaR 5%')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=30, ha='right')
plt.ylabel('USD per acre')
plt.title('Main Model: Seven-Strategy Comparison')
plt.legend()
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 18. 图2：Main vs Beta=0 CVaR

In [ ]:
xpos = np.arange(len(STRATEGIES))
width = 0.38

plt.figure(figsize=(12, 5))
plt.bar(
    xpos - width/2,
    main_summary['CVaR 5% Profit ($/acre)'],
    width,
    label='Main model',
    color='#4472C4',
)
plt.bar(
    xpos + width/2,
    robust_summary['CVaR 5% Profit ($/acre)'],
    width,
    label='July beta = 0',
    color='#70AD47',
)
plt.xticks(xpos, STRATEGIES, rotation=30, ha='right')
plt.ylabel('CVaR 5% profit (USD per acre)')
plt.title('Robustness of Downside-Risk Ranking')
plt.legend()
plt.tight_layout()
plt.show()

## 19. 主模型必须通过的检查

In [ ]:
assert len(main_results) == N_SIMULATIONS * len(STRATEGIES)
assert not main_results.duplicated(['scenario_id', 'strategy']).any()
assert main_results.isna().sum().sum() == 0
assert set(main_results['strategy']) == set(STRATEGIES)
assert main_summary['Preferred'].sum() == 1
assert main_preferred == 'Fixed 75%'

expected_main_metrics = {
    'Fixed 0%': [-1.5944606006, -338.7472297638, 0.0000],
    'Fixed 25%': [5.4810561656, -240.0045877430, 0.0000],
    'Fixed 50%': [11.9133441351, -154.8015607295, 0.0000],
    'Fixed 75%': [18.9888609010, -94.9466253290, 0.0000],
    'Fixed 100%': [25.4211488710, -118.1373768120, 0.4538],
    'Updated 50%': [12.0031094390, -155.0416200000, 0.0000],
    'Weather-signal 25/50/75%': [13.1318404060, -201.4749346266, 0.0000],
}

for strategy, expected in expected_main_metrics.items():
    row = main_summary[main_summary['strategy'] == strategy].iloc[0]
    actual = np.array([
        row['Expected Profit ($/acre)'],
        row['CVaR 5% Profit ($/acre)'],
        row['Probability Overhedged'],
    ])
    assert np.allclose(actual, expected, atol=1e-6), (strategy, actual, expected)

print('主模型检查通过。')

## 20. Robustness必须通过的检查

In [ ]:
assert len(robust_results) == N_SIMULATIONS * len(STRATEGIES)
assert not robust_results.duplicated(['scenario_id', 'strategy']).any()
assert robust_results.isna().sum().sum() == 0
assert robust_summary['Preferred'].sum() == 1
assert robust_preferred == 'Fixed 75%'
assert main_preferred == robust_preferred

expected_robust_adaptive = {
    'Updated 50%': [12.1866608770, -154.8891860000, 0.3859],
    'Weather-signal 25/50/75%': [16.7991968450, -198.9339500000, 0.3489],
}

for strategy, expected in expected_robust_adaptive.items():
    row = robust_summary[robust_summary['strategy'] == strategy].iloc[0]
    actual = np.array([
        row['Expected Profit ($/acre)'],
        row['CVaR 5% Profit ($/acre)'],
        row['Probability Margin Call Proxy'],
    ])
    assert np.allclose(actual, expected, atol=1e-6), (strategy, actual, expected)

print('July beta = 0 robustness检查通过。')

## 21. 验证教授要求的交易价格追踪

In [ ]:
for label, scenario_data, results in [
    ('Main', main_scenarios, main_results),
    ('Beta0', robust_scenarios, robust_results),
]:
    f0_repeated = np.tile(
        scenario_data['preseason_futures_usd_per_bushel'].to_numpy(),
        len(STRATEGIES),
    )
    fj_repeated = np.tile(
        scenario_data['july_futures_usd_per_bushel'].to_numpy(),
        len(STRATEGIES),
    )
    fh_repeated = np.tile(
        scenario_data['harvest_futures_usd_per_bushel'].to_numpy(),
        len(STRATEGIES),
    )

    expected_initial_pnl = (
        results['initial_contracts'].to_numpy()
        * CONTRACT_SIZE_BUSHELS
        * (f0_repeated - fh_repeated)
    )
    expected_july_pnl = (
        results['july_adjustment_contracts'].to_numpy()
        * CONTRACT_SIZE_BUSHELS
        * (fj_repeated - fh_repeated)
    )

    assert np.allclose(results['initial_futures_pnl_usd'], expected_initial_pnl)
    assert np.allclose(results['july_adjustment_pnl_usd'], expected_july_pnl)
    print(label, 'transaction-price tracking: PASS')

## 22. 保存第11课结果

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_11_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

main_results_path = OUTPUT_DIR / 'main_model_all_strategy_results_70000.csv'
main_summary_path = OUTPUT_DIR / 'main_model_strategy_summary.csv'
robust_results_path = OUTPUT_DIR / 'beta0_all_strategy_results_70000.csv'
robust_summary_path = OUTPUT_DIR / 'beta0_strategy_summary.csv'
decision_path = OUTPUT_DIR / 'step_11_final_decision.json'

main_results.to_csv(main_results_path, index=False)
main_summary.to_csv(main_summary_path, index=False)
robust_results.to_csv(robust_results_path, index=False)
robust_summary.to_csv(robust_summary_path, index=False)

decision = {
    'preferred_strategy_main_model': main_preferred,
    'preferred_strategy_july_beta_zero': robust_preferred,
    'recommendation_robust_to_july_beta_zero': bool(main_preferred == robust_preferred),
    'primary_objective': 'maximize CVaR 5% profit per acre among eligible strategies',
    'expected_profit_constraint': 'mean profit no more than 5% of abs(unhedged mean) below unhedged mean',
    'maximum_overhedge_probability': MAX_OVERHEDGE_PROBABILITY,
    'main_model_july_price_beta': MAIN_JULY_PRICE_BETA,
    'robustness_july_price_beta': 0.0,
}
decision_path.write_text(json.dumps(decision, indent=2), encoding='utf-8')

print('已保存:', main_results_path)
print('已保存:', main_summary_path)
print('已保存:', robust_results_path)
print('已保存:', robust_summary_path)
print('已保存:', decision_path)

## 23. 最终模型结论

### 推荐策略

在本项目声明的选择标准下，**Fixed 75%**是首选策略。

主模型中，它的expected profit约$18.99/acre，profit standard deviation约$52.85/acre，P5约−$67.65/acre，CVaR 5%约−$94.95/acre，over-hedging probability为0%。

Fixed 100%虽然expected profit更高，但over-hedging probability约45.38%，超过10%约束，因此不合格。

### Robustness

把解释力较弱的July price beta设为0后，首选仍然是Fixed 75%。因此最终推荐不依赖该July price coefficient。

### 正确表述

不要写“Fixed 75%在现实中一定最好”。应写：

> Under the stated data, cost, basis, price-model, and implementation assumptions, Fixed 75% provides the strongest lower-tail profit protection among eligible strategies, and the recommendation is unchanged when the weak July price-signal coefficient is set to zero.

Python分析已经完成。下一阶段是把模型整理成最终formula-driven Excel、报告框架和提交检查清单。